# 利用模型蒸馏微调模型

OpenAI **蒸馏（Distillation）** 使用较大模型的输出来微调较小的模型。改用较小的模型可以降低特定任务的延迟和成本。在本 cookbook 中，我们会将 `gpt-4o` 的输出蒸馏到 `gpt-4o-mini` 中，并将结果与未经蒸馏的通用 `gpt-4o-mini` 模型进行比较。

我们还会使用 **Structured Outputs**，借助枚举（enum）进行分类。我们将衡量结构化输出对微调模型的影响，并展示它适用于每个模型，包括蒸馏后的模型。

我们首先分析数据集，获取 4o 和 4o mini 的输出，突出两种模型在性能上的差异，然后进行蒸馏并分析蒸馏后模型的性能。

## 前提条件

我们来安装并加载依赖。
请确保已在环境中将 OpenAI API 密钥定义为 "OPENAI_API_KEY"，客户端会直接加载它。

In [8]:
! pip install openai tiktoken numpy pandas tqdm --quiet

In [9]:
import openai
import json
import tiktoken
from tqdm import tqdm
from openai import OpenAI
import numpy as np
import concurrent.futures
import pandas as pd

client = OpenAI()

## 加载并理解数据集

在本 cookbook 中，我们将从以下 Kaggle 挑战赛加载数据：[https://www.kaggle.com/datasets/zynicide/wine-reviews](https://www.kaggle.com/datasets/zynicide/wine-reviews)。

该数据集包含很多行，你可以在完整数据集上运行本 cookbook。作为一名有所偏爱的法国葡萄酒爱好者，我会把数据集缩小到法国葡萄酒，这样可以专注于更少的行和葡萄品种。

我们要解决的是一个分类问题：希望根据现有的信息（包括描述、子产区和省份，这些都会放进提示词中）预测葡萄品种。这为模型提供了大量信息。你可以删除那些会让任务明显变简单的信息（例如葡萄的产区），看看模型预测品种的效果如何。

我们先过滤掉在评论中出现次数少于五次的葡萄品种。

接下来，我们从该数据集中随机抽取 500 行作为子集进行处理。

In [10]:
df = pd.read_csv('data/winemag/winemag-data-130k-v2.csv')
df_france = df[df['country'] == 'France']

# Let's also filter out wines that have less than 5 references with their grape variety – even though we'd like to find those
# they're outliers that we don't want to optimize for that would make our enum list be too long
# and they could also add noise for the rest of the dataset on which we'd like to guess, eventually reducing our accuracy.

varieties_less_than_five_list = df_france['variety'].value_counts()[df_france['variety'].value_counts() < 5].index.tolist()
df_france = df_france[~df_france['variety'].isin(varieties_less_than_five_list)]

df_france_subset = df_france.sample(n=500)
df_france_subset.head()

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
95206,95206,France,"Full, fat, ripe, perfumed wine that is full of...",Château de Mercey Premier Cru,91,35.0,Burgundy,Mercurey,NaN,Roger Voss,@vossroger,Antonin Rodet 2010 Château de Mercey Premier C...,Pinot Noir,Antonin Rodet
66403,66403,France,"For simple Chablis, this is impressive, rich, ...",Domaine,89,26.0,Burgundy,Chablis,NaN,Roger Voss,@vossroger,William Fèvre 2005 Domaine (Chablis),Chardonnay,William Fèvre
71277,71277,France,This 50-50 blend of Marselan and Merlot opens ...,La Remise,84,13.0,France Other,Vin de France,NaN,Lauren Buzzeo,@laurbuzz,Domaine de la Mordorée 2014 La Remise Red (Vin...,Red Blend,Domaine de la Mordorée
27484,27484,France,The medium-intense nose of this solid and easy...,Authentic & Chic,86,10.0,France Other,Vin de France,NaN,Lauren Buzzeo,@laurbuzz,Romantic 2014 Authentic & Chic Cabernet Sauvig...,Cabernet Sauvignon,Romantic
124917,124917,France,"Fresh, pure notes of Conference pear peel enti...",NaN,89,30.0,Alsace,Alsace,NaN,Anne Krebiehl MW,@AnneInVino,Domaine Vincent Stoeffler 2015 Pinot Gris (Als...,Pinot Gris,Domaine Vincent Stoeffler


我们来获取所有葡萄品种，将其放入提示词以及结构化输出的枚举列表中。

In [11]:
varieties = np.array(df_france['variety'].unique()).astype('str')
varieties

array(['Gewürztraminer', 'Pinot Gris', 'Gamay',
       'Bordeaux-style White Blend', 'Champagne Blend', 'Chardonnay',
       'Petit Manseng', 'Riesling', 'White Blend', 'Pinot Blanc',
       'Alsace white blend', 'Bordeaux-style Red Blend', 'Malbec',
       'Tannat-Cabernet', 'Rhône-style Red Blend', 'Ugni Blanc-Colombard',
       'Savagnin', 'Pinot Noir', 'Rosé', 'Melon',
       'Rhône-style White Blend', 'Pinot Noir-Gamay', 'Colombard',
       'Chenin Blanc', 'Sylvaner', 'Sauvignon Blanc', 'Red Blend',
       'Chenin Blanc-Chardonnay', 'Cabernet Sauvignon', 'Cabernet Franc',
       'Syrah', 'Sparkling Blend', 'Duras', 'Provence red blend',
       'Tannat', 'Merlot', 'Malbec-Merlot', 'Chardonnay-Viognier',
       'Cabernet Franc-Cabernet Sauvignon', 'Muscat', 'Viognier',
       'Picpoul', 'Altesse', 'Provence white blend', 'Mondeuse',
       'Grenache-Syrah', 'G-S-M', 'Pinot Meunier', 'Cabernet-Syrah',
       'Vermentino', 'Marsanne', 'Colombard-Sauvignon Blanc',
       'Gros and Peti

## 生成提示词

我们来编写一个生成提示词的函数，并用列表中的第一款葡萄酒试一试。

In [12]:
def generate_prompt(row, varieties):
    # Format the varieties list as a comma-separated string
    variety_list = ', '.join(varieties)
    
    prompt = f"""
    Based on this wine review, guess the grape variety:
    This wine is produced by {row['winery']} in the {row['province']} region of {row['country']}.
    It was grown in {row['region_1']}. It is described as: "{row['description']}".
    The wine has been reviewed by {row['taster_name']} and received {row['points']} points.
    The price is {row['price']}.

    Here is a list of possible grape varieties to choose from: {variety_list}.
    
    What is the likely grape variety? Answer only with the grape variety name or blend from the list.
    """
    return prompt

# Example usage with a specific row
prompt = generate_prompt(df_france.iloc[0], varieties)
prompt

'\n    Based on this wine review, guess the grape variety:\n    This wine is produced by Trimbach in the Alsace region of France.\n    It was grown in Alsace. It is described as: "This dry and restrained wine offers spice in profusion. Balanced with acidity and a firm texture, it\'s very much for food.".\n    The wine has been reviewed by Roger Voss and received 87 points.\n    The price is 24.0.\n\n    Here is a list of possible grape varieties to choose from: Gewürztraminer, Pinot Gris, Gamay, Bordeaux-style White Blend, Champagne Blend, Chardonnay, Petit Manseng, Riesling, White Blend, Pinot Blanc, Alsace white blend, Bordeaux-style Red Blend, Malbec, Tannat-Cabernet, Rhône-style Red Blend, Ugni Blanc-Colombard, Savagnin, Pinot Noir, Rosé, Melon, Rhône-style White Blend, Pinot Noir-Gamay, Colombard, Chenin Blanc, Sylvaner, Sauvignon Blanc, Red Blend, Chenin Blanc-Chardonnay, Cabernet Sauvignon, Cabernet Franc, Syrah, Sparkling Blend, Duras, Provence red blend, Tannat, Merlot, Malbec

在运行查询之前，使用 tiktoken 估算 token 数量和补全成本。该估算不包括微调，微调成本还取决于训练集大小和训练轮数（epoch）等因素。

In [13]:
# Load encoding for the GPT-4 model
enc = tiktoken.encoding_for_model("gpt-4o")

# Initialize a variable to store the total number of tokens
total_tokens = 0

for index, row in df_france_subset.iterrows():
    prompt = generate_prompt(row, varieties)
    
    # Tokenize the input text and count tokens
    tokens = enc.encode(prompt)
    token_count = len(tokens)
    
    # Add the token count to the total
    total_tokens += token_count

print(f"Total number of tokens in the dataset: {total_tokens}")
print(f"Total number of prompts: {len(df_france_subset)}")

Total number of tokens in the dataset: 245439
Total number of prompts: 500


In [14]:
# outputing cost in $ as of 2024/10/16

gpt4o_token_price = 2.50 / 1_000_000  # $2.50 per 1M tokens
gpt4o_mini_token_price = 0.150 / 1_000_000  # $0.15 per 1M tokens

total_gpt4o_cost = gpt4o_token_price*total_tokens
total_gpt4o_mini_cost = gpt4o_mini_token_price*total_tokens

print(total_gpt4o_cost)
print(total_gpt4o_mini_cost)

0.6135975
0.03681585


## 准备用于存储补全结果的函数

由于回复必须是一组固定葡萄品种中的一个，我们将使用结构化输出把模型限制在该集合内。这样可以得到一个确定的值，用来与预期品种进行比较，并防止出现数据集以外的答案。

如果你想进一步了解 Structured Outputs，可以阅读这篇 [cookbook](https://cookbook.openai.com/examples/structured_outputs_intro) 和这份[文档指南](https://platform.openai.com/docs/guides/structured-outputs/introduction)。

In [15]:
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "grape-variety",
        "schema": {
            "type": "object",
            "properties": {
                "variety": {
                    "type": "string",
                    "enum": varieties.tolist()
                }
            },
            "additionalProperties": False,
            "required": ["variety"],
        },
        "strict": True
    }
}

要蒸馏模型，你需要存储某个模型的所有补全结果，以便将其作为参考提供给较小的模型进行微调。因此，我们在 `client.chat.completions.create` 方法中添加了 `store=True` 参数，以便存储来自 gpt-4o 的这些补全结果。

我们将存储所有补全结果（包括 4o-mini 和我们之后的微调模型），这样就能直接在 OpenAI 平台上运行 [Evals](https://platform.openai.com/docs/guides/evals)。

存储这些补全结果时，为其加上元数据标签会很有用，这样就可以在 OpenAI 平台上进行筛选，只针对你想要的那组补全结果运行蒸馏和评估。

In [16]:
# Initialize the progress index
metadata_value = "wine-distillation" # that's a funny metadata tag :-)

# Function to call the API and process the result for a single model (blocking call in this case)
def call_model(model, prompt):
    response = client.chat.completions.create(
        model=model,
        store=True,
        metadata={
            "distillation": metadata_value,
        },
        messages=[
            {
                "role": "system",
                "content": "You're a sommelier expert and you know everything about wine. You answer precisely with the name of the variety/blend."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
         response_format=response_format
    )
    return json.loads(response.choices[0].message.content.strip())['variety']

## 并行处理

由于我们要在大量行上运行，需要确保这些补全请求并行执行，为此我们使用 concurrent futures。我们会遍历 dataframe，每处理 20 行输出一次进度。所运行模型的补全结果会存储在同一个 dataframe 中，列名为 `{model}-variety`。

In [17]:
def process_example(index, row, model, df, progress_bar):
    global progress_index

    try:
        # Generate the prompt using the row
        prompt = generate_prompt(row, varieties)

        df.at[index, model + "-variety"] = call_model(model, prompt)
        
        # Update the progress bar
        progress_bar.update(1)
        
        progress_index += 1
    except Exception as e:
        print(f"Error processing model {model}: {str(e)}")

def process_dataframe(df, model):
    global progress_index
    progress_index = 1  # Reset progress index

    # Create a tqdm progress bar
    with tqdm(total=len(df), desc="Processing rows") as progress_bar:
        # Process each example concurrently using ThreadPoolExecutor
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures = {executor.submit(process_example, index, row, model, df, progress_bar): index for index, row in df.iterrows()}
            
            for future in concurrent.futures.as_completed(futures):
                try:
                    future.result()  # Wait for each example to be processed
                except Exception as e:
                    print(f"Error processing example: {str(e)}")

    return df

在处理整个 dataframe 之前，我们先试用一下调用模型的函数并检查输出。

In [18]:
answer = call_model('gpt-4o', generate_prompt(df_france_subset.iloc[0], varieties))
answer

'Pinot Noir'

太好了！我们确认可以得到葡萄品种作为输出，现在用 `gpt-4o` 和 `gpt-4o-mini` 分别处理数据集并比较结果。

In [19]:
df_france_subset = process_dataframe(df_france_subset, "gpt-4o")

Processing rows: 100%|███████████████████████████████████████████████| 500/500 [00:41<00:00, 12.09it/s]


In [20]:
df_france_subset = process_dataframe(df_france_subset, "gpt-4o-mini")

Processing rows: 100%|███████████████████████████████████████████████| 500/500 [01:31<00:00,  5.45it/s]


## 比较 gpt-4o 和 gpt-4o-mini

现在我们已经拿到了这两个模型的所有对话补全结果，接下来将它们与预期的葡萄品种进行比较，评估各自识别品种的准确率。由于只需进行简单的字符串检查，我们直接在 Python 中完成；但如果你的任务涉及更复杂的评估，可以借助 OpenAI Evals 或我们的开源评估框架。

In [21]:
models = ['gpt-4o', 'gpt-4o-mini']

def get_accuracy(model, df):
    return np.mean(df['variety'] == df[model + '-variety'])

for model in models:
    print(f"{model} accuracy: {get_accuracy(model, df_france_subset) * 100:.2f}%")

gpt-4o accuracy: 81.80%
gpt-4o-mini accuracy: 69.00%


可以看到，gpt-4o 识别葡萄品种的能力优于 4o-mini（高出 12.80%，相对 4o-mini 提升了近 20%！）。我都开始怀疑我们是不是在训练时让 gpt-4o 喝了葡萄酒！

## 将 gpt-4o 的输出蒸馏到 gpt-4o-mini

假设我们需要经常运行这种预测，希望补全更快、更便宜，同时保持这一准确率水平。如果能把 4o 的准确率蒸馏到 4o-mini 上就太好了，不是吗？我们开始吧！

现在前往 OpenAI 的已存储补全页面：[https://platform.openai.com/chat-completions](https://platform.openai.com/chat-completions)。

选择模型 gpt-4o（务必这样做，你不会想蒸馏我们运行的 4o-mini 的输出）。再选择元数据 `distillation: wine-distillation`，只获取由本 cookbook 运行并存储的补全结果。

![筛选补全结果](../images/filtering-out-completions.png)

选好补全结果后，点击右上角的 "Distill"，即可基于这些补全结果微调模型。完成后，系统会自动创建一个用于运行微调流程的文件。然后选择 `gpt-4o-mini` 作为基础模型，保留默认参数（当然你也可以修改参数或反复迭代以提升性能）。

![蒸馏弹窗](../images/distilling.png)

微调任务启动后，你可以在微调页面获取微调任务 ID，我们将用它来监控微调任务的状态，并在完成后获取微调模型的 ID。

![微调任务](../images/fine-tuning-job.png)

In [22]:
# copy paste your fine-tune job ID below
finetune_job = client.fine_tuning.jobs.retrieve("ftjob-pRyNWzUItmHpxmJ1TX7FOaWe")

if finetune_job.status == 'succeeded':
    fine_tuned_model = finetune_job.fine_tuned_model
    print('finetuned model: ' + fine_tuned_model)
else:
    print('finetuned job status: ' + finetune_job.status)

finetuned model: ft:gpt-4o-mini-2024-07-18:distillation-test:wine-distillation:AIZntSyE


## 使用蒸馏后的模型运行补全

模型微调完成后，我们就可以用它来运行补全，并将准确率与 gpt4o 和 gpt4o-mini 进行比较。
我们另取一个法国葡萄酒子集（由于我们把输出限制为法国葡萄品种且不含异常值，验证数据集也需要聚焦于此）。我们让每个模型各在 300 条数据上运行。

In [23]:
validation_dataset = df_france.sample(n=300)

models.append(fine_tuned_model)

for model in models:
    another_subset = process_dataframe(validation_dataset, model)

Processing rows: 100%|███████████████████████████████████████████████| 300/300 [00:37<00:00,  8.08it/s]


我们来比较各模型的准确率

In [24]:
for model in models:
    print(f"{model} accuracy: {get_accuracy(model, another_subset) * 100:.2f}%")

gpt-4o accuracy: 79.67%
gpt-4o-mini accuracy: 64.67%
ft:gpt-4o-mini-2024-07-18:distillation-test:wine-distillation:AIZntSyE accuracy: 79.33%


相比未经蒸馏的 gpt-4o-mini，**相对提升了近 22%！🎉**

我们的微调模型表现远超 gpt-4o-mini，而二者的基础模型相同。今后在预测葡萄品种时，我们就能以更低的成本和更低的延迟使用该模型进行推理。